<a href="https://colab.research.google.com/github/Thampi-hub/Springboard_RT/blob/main/Capstone_2/2_DataWrangling/readData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
import pandas as pd
import numpy as np
from datetime import datetime

In [84]:
#Read in JSON files (Category names):
# ca_json = pd.read_json("./Kaggle_data/CA_category_id.json")["items"].to_dict()
# us_json = pd.read_json("./Kaggle_data/US_category_id.json")["items"].to_dict()
#       --------- OR ---------
url_ca_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CA_category_id.json"
url_us_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/US_category_id.json"
ca_json = pd.read_json(url_ca_json)["items"].to_dict()
us_json = pd.read_json(url_us_json)["items"].to_dict()

cat_id  = []
cat_val = []
for idx in ca_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])
for idx in us_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])

category_mapping = pd.DataFrame({'category_id':cat_id, 'category':cat_val }).drop_duplicates()

In [85]:
#Read in CSV files (Core data files):
# ca_csv = pd.read_csv("./Kaggle_data/CAvideos.csv")
# us_csv = pd.read_csv("./Kaggle_data/USvideos.csv")
#       --------- OR ---------
url_ca_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CAvideos.csv"
url_us_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/USvideos.csv"
ca_csv = pd.read_csv(url_ca_csv)
us_csv = pd.read_csv(url_us_csv)

ca_csv['country'] = "Canada"
us_csv['country'] = "USA"

allCSV = pd.concat([ca_csv,us_csv], axis=0)
print("CSV dimensions: ", ca_csv.shape, " + ", us_csv.shape, " = ", allCSV.shape)


CSV dimensions:  (40881, 17)  +  (40949, 17)  =  (81830, 17)


In [86]:
#Merge all data:
allData = pd.merge(allCSV, category_mapping, on="category_id", how="left")

print("Dimensions: ", allData.shape, "\n\n")
print(allData.columns.values,"\n")
print(allData.dtypes)

Dimensions:  (81830, 18) 


['video_id' 'trending_date' 'title' 'channel_title' 'category_id'
 'publish_time' 'tags' 'views' 'likes' 'dislikes' 'comment_count'
 'thumbnail_link' 'comments_disabled' 'ratings_disabled'
 'video_error_or_removed' 'description' 'country' 'category'] 

video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
country                   object
category                  object
dtype: object


In [87]:
allData['trending_date'] = pd.to_datetime(allData['trending_date'], format="%y.%d.%m")

allData['publish_datetime']  = pd.to_datetime(allData['publish_time'], utc=True)
allData['publish_date']  = allData['publish_datetime'].dt.date
allData['publish_time']  = allData['publish_datetime'].dt.time


In [95]:
# Reorder columns: ID vars, followed by features
id_cols = ["video_id","publish_datetime","trending_date","title","channel_title","category","country","likes","dislikes","comment_count","views"]
new_col_order = id_cols + allData.columns[ ~allData.columns.isin(id_cols)].tolist()
allData = allData[new_col_order]

allData.columns

Index(['video_id', 'publish_datetime', 'trending_date', 'title',
       'channel_title', 'category', 'country', 'likes', 'dislikes',
       'comment_count', 'views', 'category_id', 'publish_time', 'tags',
       'thumbnail_link', 'comments_disabled', 'ratings_disabled',
       'video_error_or_removed', 'description', 'publish_date'],
      dtype='object')

In [93]:
filtered_idx = allData[allData.title!="Deleted video"].\
                sort_values(["video_id","country","publish_datetime","trending_date"]).\
                groupby(["video_id","country"])["publish_datetime"].\
                idxmax().values

filtrData = allData[allData.index.isin(filtered_idx)]
filtrData.shape

(30769, 20)

In [101]:
deleted_id = allData[allData.title=="Deleted video"]["video_id"].unique()
# allData[allData.video_id.isin(deleted_id)]

err_rem_id = allData[allData.video_error_or_removed]["video_id"].unique()
# allData[allData.video_id.isin(err_rem_id)]

uniq_err_id = ['KjSSU1trCug','0JqXITEsMy8','q8v9MvManKE','BEePFpC9qG8','1Aoc-cd9eYs','RK_B4Ez4_5Q']
allData[allData.video_id.isin(uniq_err_id)].sort_values(["video_id","country","publish_datetime","trending_date"])
# print(deleted_id, "\n\n", err_rem_id)


,video_id,publish_datetime,trending_date,title,channel_title,category,country,likes,dislikes,comment_count,views,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,publish_date
28720,0JqXITEsMy8,2018-04-05 06:56:09+00:00,2018-04-15,Coachella 2018 LIVE Channel 1 (CA),Coachella,Music,Canada,226862,5039,22284,8478721,10,06:56:09,[none],https://i.ytimg.com/vi/0JqXITEsMy8/default_liv...,False,False,True,For more cameras tune in @ https://yt.be/coach...,2018-04-05
32758,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-05,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,Canada,27216,930,2630,673534,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
33019,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-06,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,Canada,23414,1313,4466,982049,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
73639,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-05,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,28696,3766,5033,833027,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
73852,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-06,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,4800,177,1223,641055,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
74068,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-07,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,4916,197,1269,672609,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
74295,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-08,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,25441,208,952,412323,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
74501,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-09,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,6623,231,1120,251857,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
74718,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-10,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,30695,277,1033,629055,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02
75475,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-14,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,USA,32276,4594,5547,1062499,24,16:02:35,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",https://i.ytimg.com/vi/1Aoc-cd9eYs/default.jpg,False,False,True,Present day Daniel LaRusso lives a charmed lif...,2018-05-02


In [90]:
#No. of tags:
allData['n_tags'] = allData['tags'].apply(lambda x: len(x.split("|")) )

#No. of Repeats:
video_n  = allData.groupby('video_id')['video_id'].count()
vid_n_DF = pd.DataFrame({"video_id" : video_n.index,
                         "trend_rep": video_counts.values})
allData = pd.merge(allData, vid_n_DF, on="video_id", how="left")
allData.shape